# ME 313 · Lab 7.2 — An IR-thermography emissivity corrector
**Module 7 companion (Radiation).**

A thermal camera measures *radiance*, not temperature. If it assumes a blackbody ($\varepsilon=1$) but the surface is grey, 
the reported ('apparent') temperature is wrong — sometimes by tens of degrees. Correcting for emissivity is a real, 
money-on-the-line problem in inspection. You build a corrector and show how much error it removes, always **checking against $\varepsilon\sigma T^4$**.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
sig = 5.67e-8

### 1. What the camera actually sees
Received radiance $\approx \varepsilon\sigma T_s^4+(1-\varepsilon)\sigma T_{surr}^4$. 
A blackbody-assuming camera converts that to an **apparent** temperature $T_{app}=(L/\sigma)^{1/4}$.


In [ ]:
rng = np.random.default_rng(3)
def apparent_T(Ts, eps, Tsurr):
    L = eps*sig*Ts**4 + (1-eps)*sig*Tsurr**4
    return (L/sig)**0.25

N = 1500
Ts    = rng.uniform(300, 500, N)     # true surface temp (K)
eps   = rng.uniform(0.3, 0.98, N)    # emissivity
Tsurr = rng.uniform(280, 310, N)     # reflected background
Tapp  = apparent_T(Ts, eps, Tsurr)
X = np.column_stack([Tapp, eps, Tsurr]); y = Ts

### 2. Train the corrector and compare to the naive reading
Baseline = trust the camera ($T_{app}$). Corrected = model using $T_{app}$, $\varepsilon$, $T_{surr}$.


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
reg = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
pred = reg.predict(Xte)
mae_raw  = mean_absolute_error(yte, Xte[:,0])   # uncorrected apparent T
mae_corr = mean_absolute_error(yte, pred)
print(f'uncorrected MAE = {mae_raw:.1f} K')
print(f'corrected   MAE = {mae_corr:.1f} K')

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(yte, Xte[:,0], s=8, alpha=.35, label='uncorrected')
plt.scatter(yte, pred, s=8, alpha=.5, label='corrected')
lims=[yte.min(), yte.max()]; plt.plot(lims, lims, 'k--')
plt.xlabel('true T (K)'); plt.ylabel('estimated T (K)'); plt.legend(); plt.grid(alpha=.3); plt.show()

### 3. Verify the physics
Pick one test case, plug its $T_{app},\varepsilon,T_{surr}$ back into the radiance equation, and confirm the corrector's answer is consistent with $\varepsilon\sigma T^4$.


### Your turn
1. Restrict $\varepsilon\in[0.9,0.98]$ (near-black). Does the correction still matter? When is trusting the camera fine?
2. Add noise to $\varepsilon$ (you rarely know it exactly). How much does that reintroduce error?
3. **AI companion:** ask an LLM why a low-$\varepsilon$ polished-metal reading is untrustworthy, then verify with your numbers.
4. Which single input does the model rely on most? (Try dropping $T_{surr}$.)
